# 7. Entrenamiento final y evaluación en 2026

- **Regresión lineal:** sin regularización (`regParam = 0`, `elasticNetParam = 0`).
- **Random Forest:** RF-C (`numTrees = 150`, `maxDepth = 10`, `seed = 42`).

In [ ]:
import os
import json
import pandas as pd

from IPython.display import display
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Lab7_EvaluacionFinal")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

print("Versión de Spark:", spark.version)

assert spark.version.startswith("3.5."), "El laboratorio requiere Spark 3.5.x."

## Configuraciones seleccionadas

In [ ]:
PARQUET_DIR = "../working_dir/parquet"
MODELOS_DIR = "../working_dir/modelos"

with open(f"{MODELOS_DIR}/lr_resumen_validacion.json", encoding="utf-8") as archivo:
    resumen_lr = json.load(archivo)

with open(f"{MODELOS_DIR}/rf_resumen_validacion.json", encoding="utf-8") as archivo:
    resumen_rf = json.load(archivo)

config_lr = resumen_lr["configuracion"]
config_rf = resumen_rf["configuracion"]
SEMILLA = resumen_rf["seed"]

NUMERICAS = resumen_lr["predictores_numericos"]
CATEGORICAS = resumen_lr["predictores_categoricos"]
OBJETIVO = resumen_lr["objetivo"]

assert NUMERICAS == resumen_rf["predictores_numericos"]
assert CATEGORICAS == resumen_rf["predictores_categoricos"]

display(pd.DataFrame([
    {"algoritmo": "Regresión lineal", **config_lr},
    {"algoritmo": "Random Forest", **config_rf, "seed": SEMILLA},
]))

print("Predictores numéricos:", NUMERICAS)
print("Predictores categóricos:", CATEGORICAS)

## Datos de entrenamiento final y de prueba

In [ ]:
COLUMNAS_MODELO = [
    "periodo_archivo",
    "NUM_HOGAR",
    "NUM_PERSONA",
    OBJETIVO,
    *NUMERICAS,
    *CATEGORICAS,
    "nivel_educativo_etiqueta",
    "dominio_etiqueta",
]

train_final = (
    spark.read.parquet(f"{PARQUET_DIR}/eneic_2025_preparado")
    .select(*COLUMNAS_MODELO)
    .cache()
)

test_2026 = (
    spark.read.parquet(f"{PARQUET_DIR}/eneic_2026_preparado")
    .select(*COLUMNAS_MODELO)
    .cache()
)

n_train_final = train_final.count()
n_test = test_2026.count()

print(f"Entrenamiento final (2025): {n_train_final:,} registros")
print(f"Prueba (2026T1):            {n_test:,} registros")

(
    train_final.withColumn("conjunto", F.lit("Entrenamiento final"))
    .unionByName(test_2026.withColumn("conjunto", F.lit("Prueba")))
    .groupBy("conjunto", "periodo_archivo")
    .count()
    .orderBy("periodo_archivo")
    .show(truncate=False)
)

### Verificación de la preparación del conjunto de prueba

In [ ]:
def resumen_reglas(df):
    return df.agg(
        F.count("*").alias("registros"),
        F.sum(
            sum(F.col(c).isNull().cast("int") for c in [OBJETIVO, *NUMERICAS, *CATEGORICAS])
        ).alias("valores_nulos"),
        F.sum((F.col(OBJETIVO) <= 0).cast("int")).alias("salario_no_positivo"),
        F.sum((F.col("edad") < 15).cast("int")).alias("edad_menor_15"),
        F.sum((F.col("antiguedad") > F.col("edad")).cast("int")).alias("antiguedad_mayor_edad"),
        F.sum(((F.col("horas_semanales") <= 0) | (F.col("horas_semanales") > 168)).cast("int")).alias("horas_fuera_rango"),
    ).toPandas()

display(pd.concat([
    resumen_reglas(train_final).assign(conjunto="Entrenamiento final"),
    resumen_reglas(test_2026).assign(conjunto="Prueba 2026"),
]).set_index("conjunto"))

# Categorías de 2026 que no aparecieron en el entrenamiento
for columna in CATEGORICAS:
    nuevas = (
        test_2026.select(columna).distinct()
        .join(train_final.select(columna).distinct(), on=columna, how="left_anti")
        .count()
    )
    print(f"{columna}: {nuevas} categorías de 2026 que no están en el entrenamiento")

## Pipelines y métricas

In [ ]:
def crear_etapas_preparacion():
    indexadores = [
        StringIndexer(
            inputCol=columna,
            outputCol=f"{columna}_idx",
            handleInvalid="keep",
            stringOrderType="alphabetAsc"
        )
        for columna in CATEGORICAS
    ]

    codificador = OneHotEncoder(
        inputCols=[f"{columna}_idx" for columna in CATEGORICAS],
        outputCols=[f"{columna}_ohe" for columna in CATEGORICAS],
        dropLast=True
    )

    ensamblador = VectorAssembler(
        inputCols=NUMERICAS + [f"{columna}_ohe" for columna in CATEGORICAS],
        outputCol="features",
        handleInvalid="error"
    )

    return indexadores + [codificador, ensamblador]


def crear_pipeline_lr(reg_param, elastic_net_param):
    regresion = LinearRegression(
        featuresCol="features",
        labelCol=OBJETIVO,
        predictionCol="prediction",
        standardization=True,
        regParam=reg_param,
        elasticNetParam=elastic_net_param,
        maxIter=200,
        tol=1e-6
    )
    return Pipeline(stages=crear_etapas_preparacion() + [regresion])


def crear_pipeline_rf(num_arboles, prof_max, semilla):
    bosque = RandomForestRegressor(
        featuresCol="features",
        labelCol=OBJETIVO,
        predictionCol="prediction",
        numTrees=num_arboles,
        maxDepth=prof_max,
        seed=semilla
    )
    return Pipeline(stages=crear_etapas_preparacion() + [bosque])


evaluadores = {
    nombre: RegressionEvaluator(labelCol=OBJETIVO, predictionCol="prediction", metricName=metrica)
    for nombre, metrica in [("MAE", "mae"), ("RMSE", "rmse"), ("R2", "r2")]
}

def calcular_metricas(predicciones):
    return {
        nombre: evaluador.evaluate(predicciones)
        for nombre, evaluador in evaluadores.items()
    }

## Entrenamiento final con todo 2025

In [ ]:
modelo_final_lr = crear_pipeline_lr(
    reg_param=config_lr["regParam"],
    elastic_net_param=config_lr["elasticNetParam"]
).fit(train_final)

modelo_final_rf = crear_pipeline_rf(
    num_arboles=config_rf["numTrees"],
    prof_max=config_rf["maxDepth"],
    semilla=SEMILLA
).fit(train_final)

# Modelo de referencia: la media del salario del entrenamiento final
media_train_final = train_final.agg(F.avg(OBJETIVO)).first()[0]

print(f"Media del salario en el entrenamiento final: Q{media_train_final:,.2f}")
print("Coeficientes de la regresión lineal:", len(modelo_final_lr.stages[-1].coefficients))
print("Árboles del Random Forest:", modelo_final_rf.stages[-1].getNumTrees)

## Predicciones sobre I de 2026

In [ ]:
CLAVE = ["periodo_archivo", "NUM_HOGAR", "NUM_PERSONA"]

pred_lr = modelo_final_lr.transform(test_2026).select(*CLAVE, F.col("prediction").alias("pred_lr"))
pred_rf = modelo_final_rf.transform(test_2026).select(*CLAVE, F.col("prediction").alias("pred_rf"))

predicciones_2026 = (
    test_2026
    .withColumn("pred_referencia", F.lit(float(media_train_final)))
    .join(pred_lr, on=CLAVE, how="inner")
    .join(pred_rf, on=CLAVE, how="inner")
    .cache()
)

n_pred = predicciones_2026.count()
nulos_pred = predicciones_2026.filter(
    F.col("pred_lr").isNull() | F.col("pred_rf").isNull()
    | F.isnan("pred_lr") | F.isnan("pred_rf")
).count()

assert n_pred == n_test, "Los modelos no se evaluaron sobre los mismos registros."
assert nulos_pred == 0, "Hay predicciones nulas."
assert predicciones_2026.select(CLAVE).distinct().count() == n_test, "Hay claves duplicadas."

print(f"Registros de prueba con predicción de los tres modelos: {n_pred:,} de {n_test:,}")

predicciones_2026.select(
    *CLAVE, OBJETIVO, "pred_referencia", "pred_lr", "pred_rf"
).show(5, truncate=False)

## Métricas de prueba (I de 2026)

In [ ]:
MODELOS = {
    "Referencia: media de entrenamiento": "pred_referencia",
    f"Regresión lineal ({config_lr['configuracion']})": "pred_lr",
    f"Random Forest ({config_rf['configuracion']})": "pred_rf",
}

filas = []
for nombre, columna in MODELOS.items():
    metricas = calcular_metricas(predicciones_2026.withColumn("prediction", F.col(columna)))
    filas.append({"modelo": nombre, "registros": n_pred, **metricas})

tabla_prueba = pd.DataFrame(filas)
rmse_ref_prueba = tabla_prueba.loc[0, "RMSE"]
mae_ref_prueba = tabla_prueba.loc[0, "MAE"]
tabla_prueba["reduccion_MAE_pct"] = 100 * (1 - tabla_prueba["MAE"] / mae_ref_prueba)
tabla_prueba["reduccion_RMSE_pct"] = 100 * (1 - tabla_prueba["RMSE"] / rmse_ref_prueba)

display(tabla_prueba.round(4))

## Validación (IV 2025) frente a prueba (I 2026)

In [ ]:
train_t1_t3 = train_final.filter(F.col("periodo_archivo").isin("2025T1", "2025T2", "2025T3")).cache()
validacion_t4 = train_final.filter(F.col("periodo_archivo") == "2025T4").cache()

modelo_val_lr = crear_pipeline_lr(config_lr["regParam"], config_lr["elasticNetParam"]).fit(train_t1_t3)
modelo_val_rf = crear_pipeline_rf(config_rf["numTrees"], config_rf["maxDepth"], SEMILLA).fit(train_t1_t3)
media_t1_t3 = train_t1_t3.agg(F.avg(OBJETIVO)).first()[0]

# Verificación: deben coincidir con las métricas guardadas en las actividades 5 y 6
reproduccion = pd.DataFrame([
    {"modelo": "Regresión lineal", "RMSE_guardado": resumen_lr["metricas_validacion_lr"]["RMSE"],
     "RMSE_reproducido": calcular_metricas(modelo_val_lr.transform(validacion_t4))["RMSE"]},
    {"modelo": "Random Forest", "RMSE_guardado": resumen_rf["metricas_validacion_rf"]["RMSE"],
     "RMSE_reproducido": calcular_metricas(modelo_val_rf.transform(validacion_t4))["RMSE"]},
])
reproduccion["diferencia"] = reproduccion["RMSE_reproducido"] - reproduccion["RMSE_guardado"]
display(reproduccion.round(4))

In [ ]:
filas_comparacion = []
for algoritmo, clave_json, resumen, modelo_val in [
    ("Referencia", "metricas_validacion_referencia", resumen_lr, None),
    ("Regresión lineal", "metricas_validacion_lr", resumen_lr, modelo_val_lr),
    ("Random Forest", "metricas_validacion_rf", resumen_rf, modelo_val_rf),
]:
    if modelo_val is None:
        pred_val_2026 = test_2026.withColumn("prediction", F.lit(float(media_t1_t3)))
    else:
        pred_val_2026 = modelo_val.transform(test_2026)
    assert pred_val_2026.count() == n_test

    fila_final = tabla_prueba.loc[tabla_prueba["modelo"].str.startswith(algoritmo)].iloc[0]
    for escenario, metricas in [
        ("Validación IV-2025 (entrenado T1–T3)", resumen[clave_json]),
        ("Prueba I-2026 (entrenado T1–T3)", calcular_metricas(pred_val_2026)),
        ("Prueba I-2026 (entrenado T1–T4, final)", fila_final[["MAE", "RMSE", "R2"]].to_dict()),
    ]:
        filas_comparacion.append({"algoritmo": algoritmo, "escenario": escenario, **metricas})

tabla_val_prueba = pd.DataFrame(filas_comparacion)
display(tabla_val_prueba.round(4))

## Interpretación

**Resultados en la prueba (I de 2026, 13,258 registros, los mismos para los tres modelos).**

| Modelo | MAE | RMSE | R² | Reducción de RMSE vs. referencia |
|---|---:|---:|---:|---:|
| Referencia (media de 2025) | Q1,718.09 | Q2,871.42 | -0.003 | — |
| Regresión lineal (sin regularización) | Q1,240.71 | Q2,163.75 | 0.431 | 24.6 % |
| Random Forest (RF-C) | **Q1,103.88** | **Q1,959.95** | **0.533** | **31.7 %** |

## Guardar los modelos finales y las predicciones de prueba

In [ ]:
RUTA_FINAL_LR = f"{MODELOS_DIR}/lr_final_2025"
RUTA_FINAL_RF = f"{MODELOS_DIR}/rf_final_2025"
RUTA_PREDICCIONES = f"{PARQUET_DIR}/predicciones_2026"

modelo_final_lr.write().overwrite().save(RUTA_FINAL_LR)
modelo_final_rf.write().overwrite().save(RUTA_FINAL_RF)
predicciones_2026.write.mode("overwrite").parquet(RUTA_PREDICCIONES)

resumen_final = {
    "periodos_entrenamiento": ["2025T1", "2025T2", "2025T3", "2025T4"],
    "periodo_prueba": "2026T1",
    "registros_entrenamiento": n_train_final,
    "registros_prueba": n_test,
    "configuracion_lr": config_lr,
    "configuracion_rf": {**config_rf, "seed": SEMILLA},
    "media_entrenamiento": float(media_train_final),
    "metricas_prueba": {
        fila["modelo"]: {m: float(fila[m]) for m in ["MAE", "RMSE", "R2"]}
        for _, fila in tabla_prueba.iterrows()
    },
    "version_spark": spark.version,
}

with open(f"{MODELOS_DIR}/resumen_prueba_2026.json", "w", encoding="utf-8") as archivo:
    json.dump(resumen_final, archivo, ensure_ascii=False, indent=2)

print("Modelos finales:", RUTA_FINAL_LR, "|", RUTA_FINAL_RF)
print("Predicciones de prueba:", RUTA_PREDICCIONES, "->",
      spark.read.parquet(RUTA_PREDICCIONES).count(), "registros")